# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at the URL below.

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install -U mlcroissant[full]

## 1. Data Loading
Load dataset metadata and examine essential details using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, not a dict
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the field and column structure using Croissant metadata.

In [ ]:
# List all record sets and their fields
record_sets = metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    Field: {field.name}, @id: {field.id}, Data type: {field.data_type}")
        for col in field.columns:
            print(f"      Column: {col.name}, @id: {col.id}, Source: {col.source}")
    print("\n")
# Store the main record set @id for extraction
main_record_set_id = record_sets[0].id if len(record_sets) > 0 else None

## 3. Data Extraction
Extract the data for each record set into a pandas DataFrame for further analysis. All record set, field, and column references use their Croissant `@id`s.

In [ ]:
dataframes = {}
rs_ids = [rs.id for rs in record_sets]
print("Extracting records from each record set...")
for rsid in rs_ids:
    print(f"  Extracting {rsid} ...")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
# Display columns of the main record set
if main_record_set_id is not None:
    print(f"Columns in {main_record_set_id}: \n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group by key fields. Use only Croissant `@id` names for fields/columns (see the overview section output).

In [ ]:
# Select a numeric field and group field by their @id as displayed above
# Update these to match actual field @id values from step 2 output, e.g., 'accession', 'description', 'mw'
record_set_id = main_record_set_id
# For demonstration, let's try common field names. Replace with exact @id as needed.
possible_numeric_fields = [
    'mw', 'molecular_weight', 'peptide_count', 'normalized_abundance', 'coverage_percentage', 'cr:field:mw', 'cr:field:molecular_weight'
]
df = dataframes[record_set_id]
numeric_field = None
for col in df.columns:
    if col.lower() in possible_numeric_fields or 'mw' in col.lower() or 'abundance' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    # Fallback: Try to find the first float/integer looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
print(f"Using numeric field: {numeric_field}")

# Filtering and normalization
if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records in '{numeric_field}' > {threshold:.2f}: {len(filtered_df)} out of {len(df)}")

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a categorical field (e.g., 'description' or 'sample')
    group_field_candidates = [
        'sample', 'condition', 'description', 'protein', 'cr:field:sample', 'cr:field:description', 'cr:field:protein'
    ]
    group_field = None
    for col in df.columns:
        if col.lower() in group_field_candidates or 'description' in col.lower():
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped means for '{numeric_field}' by '{group_field}':")
        print(grouped_df[[numeric_field]].head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Plot data distributions and relationships for selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Plot histogram for numeric field
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # If normalized, plot scatter or boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable columns for numeric visualizations.")

## 6. Conclusion

- This notebook demonstrated basic usage of `mlcroissant` for inspecting and prepping data following the FAIR principles.
- You can extend analysis by referring to each dataset element (record set, field, column) by its Croissant `@id` as enumerated above.
- For more detailed processing, refer to the field and column types given in the overview and the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
